# HackBlox 2026 â€” 3LC Ã— Scene Classification Challenge
## Data-Centric AI with Active Learning | ResNet-18 From Scratch
**Classes:** buildings(0), forest(1), glacier(2), mountain(3), sea(4), street(5)

## Section 1 â€” Setup & Imports

In [ ]:
# â”€â”€ Install required packages â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

# 3LC pinned version as required by competition
pip_install(
    '--index-url', 'https://pypi.3lc.ai/public/repositories/releases-public',
    '--extra-index-url', 'https://pypi.org/simple',
    '3lc==2.22.3'
)
pip_install('umap-learn', 'scikit-learn', 'pandas', 'matplotlib', 'seaborn', 'tqdm')
print('Packages ready.')

In [ ]:
# â”€â”€ Core imports â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os, sys, glob, json, random, warnings, copy, time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from PIL import Image

from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print('Imports OK.')

## Section 2 â€” Reproducibility

In [ ]:
# â”€â”€ Global config & seed â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
CFG = {
    'seed'           : 42,
    'num_classes'    : 6,
    'class_names'    : ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street'],
    'img_size'       : 150,
    'batch_size'     : 32,
    'num_workers'    : 2,
    'lr'             : 1e-3,
    'weight_decay'   : 1e-4,
    'epochs_base'    : 40,
    'epochs_finetune': 25,
    'patience'       : 8,
    'max_samples'    : 3000,
    'stage_targets'  : [600, 1400, 2200, 3000],
    'output_dir'     : '/kaggle/working',
}

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()

print(f'Seed    : {CFG["seed"]}')
print(f'Device  : {DEVICE}')
print(f'AMP     : {USE_AMP}')
print(f'Config  : {json.dumps(CFG, indent=2)}')

## Section 3 â€” Detect Dataset

In [ ]:
# â”€â”€ Auto-detect Kaggle input directory â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def find_dataset_root(kaggle_input: str = '/kaggle/input') -> Path:
    """
    Walk /kaggle/input looking for a folder that contains
    both 'train' and 'test' subdirectories.
    Falls back to local 'data' folder for offline dev.
    """
    base = Path(kaggle_input)
    if base.exists():
        # depth-2 search: /kaggle/input/<slug>/<maybe-subdir>
        for candidate in sorted(base.rglob('train')):
            root = candidate.parent
            if (root / 'train').is_dir() and (root / 'test').is_dir():
                return root
    # Local fallback
    local = Path('data')
    if local.exists() and (local / 'train').is_dir():
        return local
    raise FileNotFoundError('Cannot locate dataset root. Check /kaggle/input structure.')


DATASET_ROOT = find_dataset_root()
TRAIN_DIR    = DATASET_ROOT / 'train'
VAL_DIR      = DATASET_ROOT / 'val'
TEST_DIR     = DATASET_ROOT / 'test'

print(f'Dataset root : {DATASET_ROOT}')
print(f'Train path   : {TRAIN_DIR}')
print(f'Val path     : {VAL_DIR}')
print(f'Test path    : {TEST_DIR}')

## Section 4 â€” Dataset Statistics

In [ ]:
# â”€â”€ Count images in each split â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
CLASS_NAMES = CFG['class_names']

def count_images_by_class(folder: Path, classes=CLASS_NAMES):
    counts = {}
    for cls in classes:
        cls_dir = folder / cls
        if cls_dir.is_dir():
            counts[cls] = len(list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')))
        else:
            counts[cls] = 0
    return counts

# Labeled seed images (100 per class = 600 total)
seed_counts = count_images_by_class(TRAIN_DIR)
print('Labeled training images per class:')
for cls, n in seed_counts.items():
    print(f'  {cls:12s}: {n}')
print(f'  TOTAL SEED : {sum(seed_counts.values())}')

# Unlabeled pool
undef_dir = TRAIN_DIR / 'undefined'
n_undef = 0
if undef_dir.is_dir():
    # flat folder
    n_undef = len(list(undef_dir.glob('*.jpg')) + list(undef_dir.glob('*.png')))
    # also check subdirs (some versions nest by numeric subclass)
    if n_undef == 0:
        sub_counts = [len(list(s.glob('*.jpg')) + list(s.glob('*.png')))
                      for s in undef_dir.iterdir() if s.is_dir()]
        n_undef = sum(sub_counts)
print(f'\nUnlabeled pool (undefined): {n_undef}')

# Validation
val_counts = count_images_by_class(VAL_DIR)
print('\nValidation images per class:')
for cls, n in val_counts.items():
    print(f'  {cls:12s}: {n}')
n_val = sum(val_counts.values())
print(f'  TOTAL VAL  : {n_val}')

# Test (flat)
n_test = len(list(TEST_DIR.glob('*.jpg')) + list(TEST_DIR.glob('*.png')))
print(f'\nTest images (hidden labels): {n_test}')

## Section 5 â€” Data Visualization

In [ ]:
# â”€â”€ Sample images from each class â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for col, cls in enumerate(CLASS_NAMES):
    cls_dir = TRAIN_DIR / cls
    imgs = sorted(cls_dir.glob('*.jpg'))[:2] if cls_dir.is_dir() else []
    for row in range(2):
        ax = axes[row, col]
        if row < len(imgs):
            ax.imshow(Image.open(imgs[row]))
        ax.axis('off')
        if row == 0:
            ax.set_title(f'{cls}\n({seed_counts.get(cls,0)} imgs)', fontsize=9)
plt.suptitle('Sample Images â€” Labeled Training Set', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{CFG["output_dir"]}/sample_images.png', bbox_inches='tight', dpi=120)
plt.show()

# â”€â”€ Class distribution bar chart â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (title, counts) in zip(axes, [('Train (seed)', seed_counts), ('Validation', val_counts)]):
    ax.bar(counts.keys(), counts.values(), color=plt.cm.Set2.colors[:6])
    ax.set_title(title)
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(f'{CFG["output_dir"]}/class_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

## Section 6 â€” Transforms & Dataset Helpers

In [ ]:
# â”€â”€ Image transforms â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
IMG_SIZE = CFG['img_size']

# ImageNet-style mean/std (still valid even without pretrained weights â€”
# it normalises the pixel range to a sensible scale for random-init training)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(p=0.1),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomRotation(15),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

# Transform without normalisation â€” for embedding extraction consistency
embed_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

print('Transforms defined.')

In [ ]:
# â”€â”€ Dataset classes â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

class LabeledDataset(Dataset):
    """Labeled images from a class-structured directory."""
    def __init__(self, root: Path, classes=CLASS_NAMES, transform=None):
        self.samples = []   # (path, label)
        self.transform = transform
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for cls in classes:
            cls_dir = root / cls
            if not cls_dir.is_dir():
                continue
            for img_path in sorted(cls_dir.glob('*.jpg')):
                self.samples.append((str(img_path), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


class UnlabeledDataset(Dataset):
    """Unlabeled images from a flat directory (undefined pool)."""
    def __init__(self, root: Path, transform=None):
        self.paths = []
        self.transform = transform
        if root.is_dir():
            self.paths = sorted(
                list(root.glob('*.jpg')) +
                list(root.glob('*.png'))
            )
            # also collect from subdirs if any
            if not self.paths:
                for sub in sorted(root.iterdir()):
                    if sub.is_dir():
                        self.paths += sorted(sub.glob('*.jpg'))
        self.paths = [str(p) for p in self.paths]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.paths[idx]


class TestDataset(Dataset):
    """Flat test folder â€” returns (tensor, image_id)."""
    def __init__(self, root: Path, transform=None):
        self.paths = sorted(
            list(root.glob('*.jpg')) + list(root.glob('*.png'))
        )
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img_id = Path(path).stem
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, img_id


class CombinedLabeledDataset(Dataset):
    """
    Combines the original labeled seed set with pseudo-labeled
    samples selected by the active-learning pipeline.
    Each entry: (path, label).
    """
    def __init__(self, samples, transform=None):
        """
        samples: list of (path_str, int_label)
        """
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


# Build base datasets (no transform yet â€” assigned per use)
seed_ds_train = LabeledDataset(TRAIN_DIR, transform=train_transform)
seed_ds_eval  = LabeledDataset(TRAIN_DIR, transform=val_transform)
val_ds        = LabeledDataset(VAL_DIR,   transform=val_transform)
undef_ds      = UnlabeledDataset(undef_dir, transform=embed_transform)
test_ds       = TestDataset(TEST_DIR, transform=val_transform)

print(f'Seed train samples : {len(seed_ds_train)}')
print(f'Validation samples : {len(val_ds)}')
print(f'Unlabeled pool     : {len(undef_ds)}')
print(f'Test images        : {len(test_ds)}')

## Section 7 â€” Baseline ResNet-18 (from scratch)

In [ ]:
# â”€â”€ ResNet-18 building blocks â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3,
                               stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)

        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out = self.relu(out + identity)
        return out


class ResNet18(nn.Module):
    """
    ResNet-18 built entirely from scratch â€” NO pretrained weights.
    Provides `embed()` for penultimate-layer feature extraction.
    """
    def __init__(self, num_classes: int = 6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(64,  64,  2, stride=1)
        self.layer2 = self._make_layer(64,  128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(512, num_classes)
        self._init_weights()

    def _make_layer(self, in_ch, out_ch, n_blocks, stride):
        layers = [BasicBlock(in_ch, out_ch, stride=stride)]
        for _ in range(1, n_blocks):
            layers.append(BasicBlock(out_ch, out_ch, stride=1))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def embed(self, x):
        """Return 512-d penultimate embedding (before fc)."""
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)   # (B, 512)

    def forward(self, x):
        return self.fc(self.embed(x))


def make_model(num_classes=6):
    model = ResNet18(num_classes=num_classes)
    model = model.to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'ResNet-18 from scratch | trainable params: {n_params:,}')
    return model


# Quick smoke test
_ = make_model()
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    out = _(dummy)
print(f'Output shape: {out.shape}  â€” OK')
del _, dummy, out

## Section 8 â€” Training & Evaluation Utilities

In [ ]:
# â”€â”€ Training loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        preds       = logits.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    loss_avg = total_loss / total
    acc      = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return loss_avg, acc, macro_f1, all_preds, all_labels


def full_train(
    model, train_loader, val_loader,
    epochs, lr, weight_decay,
    save_path, patience=8,
    stage_label=''
):
    """
    Full training run with AdamW + CosineAnnealingLR + early stopping.
    Returns best val accuracy.
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.01)
    scaler    = GradScaler(enabled=USE_AMP)

    best_val_acc  = 0.0
    best_f1       = 0.0
    no_improve    = 0
    history       = []

    print(f'\n{"="*60}')
    print(f' Training {stage_label} | epochs={epochs} | lr={lr}')
    print(f'{"="*60}')

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        vl_loss, vl_acc, vl_f1, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history.append(dict(epoch=epoch, tr_loss=tr_loss, tr_acc=tr_acc,
                            vl_loss=vl_loss, vl_acc=vl_acc, vl_f1=vl_f1))

        improved = vl_acc > best_val_acc
        if improved:
            best_val_acc = vl_acc
            best_f1      = vl_f1
            torch.save(model.state_dict(), save_path)
            no_improve = 0
            tag = ' âœ“ saved'
        else:
            no_improve += 1
            tag = f' (no improve {no_improve}/{patience})'

        if epoch % 5 == 0 or epoch == 1 or improved:
            print(f'Ep {epoch:3d}/{epochs} | '
                  f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | '
                  f'vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} f1={vl_f1:.4f} | '
                  f'{time.time()-t0:.1f}s{tag}')

        if no_improve >= patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    print(f'Best val acc: {best_val_acc:.4f} | Best macro F1: {best_f1:.4f}')
    return best_val_acc, best_f1, history


def plot_history(history, title='Training History', save_path=None):
    epochs   = [h['epoch']  for h in history]
    tr_loss  = [h['tr_loss']  for h in history]
    vl_loss  = [h['vl_loss']  for h in history]
    tr_acc   = [h['tr_acc']   for h in history]
    vl_acc   = [h['vl_acc']   for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, tr_loss, label='Train')
    axes[0].plot(epochs, vl_loss, label='Val')
    axes[0].set_title(f'{title} â€” Loss')
    axes[0].legend()

    axes[1].plot(epochs, tr_acc, label='Train')
    axes[1].plot(epochs, vl_acc, label='Val')
    axes[1].set_title(f'{title} â€” Accuracy')
    axes[1].legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=120)
    plt.show()


def print_classification_report(labels, preds, class_names=CLASS_NAMES):
    print(classification_report(labels, preds, target_names=class_names, digits=4))


def plot_confusion_matrix(labels, preds, class_names=CLASS_NAMES, title='Confusion Matrix', save_path=None):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=120)
    plt.show()
    return cm


print('Training utilities defined.')

## Section 9 â€” Baseline Training (600 labeled samples)

In [ ]:
# â”€â”€ DataLoaders â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
set_seed(CFG['seed'])

LOADER_KW = dict(num_workers=CFG['num_workers'], pin_memory=torch.cuda.is_available())

train_loader_600 = DataLoader(
    seed_ds_train,
    batch_size=CFG['batch_size'],
    shuffle=True,
    **LOADER_KW,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG['batch_size'] * 2,
    shuffle=False,
    **LOADER_KW,
)

print(f'Train batches: {len(train_loader_600)} | Val batches: {len(val_loader)}')

In [ ]:
# â”€â”€ Train baseline on 600 seed images â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
set_seed(CFG['seed'])
model_600 = make_model()

save_600 = f'{CFG["output_dir"]}/model_600.pth'

best_acc_600, best_f1_600, history_600 = full_train(
    model_600, train_loader_600, val_loader,
    epochs=CFG['epochs_base'],
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay'],
    save_path=save_600,
    patience=CFG['patience'],
    stage_label='Stage-0 | 600 samples',
)

plot_history(history_600, title='Baseline (600)', save_path=f'{CFG["output_dir"]}/history_600.png')

## Section 10 â€” Baseline Evaluation

In [ ]:
# â”€â”€ Load best baseline weights & evaluate â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
model_600.load_state_dict(torch.load(save_600, map_location=DEVICE))
criterion_eval = nn.CrossEntropyLoss()

vl_loss, vl_acc, vl_f1, preds_600, labels_600 = evaluate(model_600, val_loader, criterion_eval)

print('\n=== Baseline (600 samples) â€” Validation Results ===')
print(f'  Val Accuracy : {vl_acc:.4f}')
print(f'  Val Loss     : {vl_loss:.4f}')
print(f'  Macro F1     : {vl_f1:.4f}')
print()
print_classification_report(labels_600, preds_600)

cm_600 = plot_confusion_matrix(
    labels_600, preds_600,
    title='Baseline Confusion Matrix (600 samples)',
    save_path=f'{CFG["output_dir"]}/cm_600.png',
)

## Section 11 â€” Embedding Extraction

In [ ]:
# â”€â”€ Extract 512-d embeddings from penultimate ResNet-18 layer â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

@torch.no_grad()
def extract_embeddings(model, dataset, batch_size=64, return_paths=False):
    """
    Returns:
        embs   : (N, 512) numpy array
        labels : (N,) numpy array of int  (or None for unlabeled)
        paths  : list of str (if return_paths and dataset yields paths)
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=CFG['num_workers'], pin_memory=torch.cuda.is_available())
    model.eval()
    embs_list, labels_list, paths_list = [], [], []

    for batch in tqdm(loader, desc='Extracting embeddings', leave=False):
        if len(batch) == 2:
            imgs, meta = batch
        else:
            imgs = batch
            meta = None

        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            e = model.embed(imgs)
        embs_list.append(e.cpu().float().numpy())

        if meta is not None:
            if isinstance(meta, torch.Tensor):
                labels_list.extend(meta.tolist())
            else:
                paths_list.extend(list(meta))

    embs = np.concatenate(embs_list, axis=0)
    labels = np.array(labels_list) if labels_list else None
    return (embs, labels, paths_list) if return_paths else (embs, labels)


# Labeled seed embeddings
seed_ds_embed = LabeledDataset(TRAIN_DIR, transform=embed_transform)
seed_embs, seed_labels = extract_embeddings(model_600, seed_ds_embed)
print(f'Seed embeddings shape : {seed_embs.shape}')

# Validation embeddings
val_ds_embed = LabeledDataset(VAL_DIR, transform=embed_transform)
val_embs, val_labels = extract_embeddings(model_600, val_ds_embed)
print(f'Val  embeddings shape : {val_embs.shape}')

# Unlabeled pool embeddings
undef_ds_embed = UnlabeledDataset(undef_dir, transform=embed_transform)
undef_embs, _, undef_paths = extract_embeddings(model_600, undef_ds_embed, return_paths=True)
print(f'Pool embeddings shape : {undef_embs.shape}')

## Section 12 â€” UMAP / PCA Visualization

In [ ]:
# â”€â”€ Dimensionality reduction for visualization â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def reduce_embeddings(embs: np.ndarray, n_components: int = 2,
                      method: str = 'auto', seed: int = 42):
    """
    Reduce to 2D with UMAP (preferred) or PCA (fallback).
    method: 'umap' | 'pca' | 'auto'
    """
    scaler = StandardScaler()
    X = scaler.fit_transform(embs)

    if method == 'auto':
        try:
            import umap
            method = 'umap'
        except ImportError:
            method = 'pca'

    if method == 'umap':
        import umap as umap_module
        reducer = umap_module.UMAP(n_components=n_components, random_state=seed,
                                   n_neighbors=15, min_dist=0.1, metric='cosine')
        print('Running UMAP...')
    else:
        reducer = PCA(n_components=n_components, random_state=seed)
        print('Running PCA...')

    reduced = reducer.fit_transform(X)
    print(f'Reduced shape: {reduced.shape} via {method.upper()}')
    return reduced, method


# Combine seed + a subsample of unlabeled for joint UMAP
N_UNDEF_SAMPLE = min(1000, len(undef_embs))   # sample for speed
rng = np.random.default_rng(CFG['seed'])
undef_idx_sample = rng.choice(len(undef_embs), N_UNDEF_SAMPLE, replace=False)

all_embs_for_reduce = np.concatenate([
    seed_embs,
    undef_embs[undef_idx_sample],
], axis=0)

all_labels_for_reduce = np.concatenate([
    seed_labels,
    np.full(N_UNDEF_SAMPLE, -1),   # -1 = unlabeled
])

reduced_2d, dim_method = reduce_embeddings(all_embs_for_reduce, method='auto')

# Plot
COLOR_MAP = plt.cm.get_cmap('tab10', 6)
COLORS    = [COLOR_MAP(i) for i in range(6)]

fig, ax = plt.subplots(figsize=(10, 8))
n_seed = len(seed_embs)

# Unlabeled pool (grey)
ax.scatter(reduced_2d[n_seed:, 0], reduced_2d[n_seed:, 1],
           c='lightgray', s=8, alpha=0.4, label='Unlabeled pool')

# Labeled seed by class
for i, cls in enumerate(CLASS_NAMES):
    mask = seed_labels == i
    ax.scatter(reduced_2d[:n_seed][mask, 0], reduced_2d[:n_seed][mask, 1],
               c=[COLORS[i]], s=25, alpha=0.85, label=cls)

ax.set_title(f'{dim_method.upper()} â€” Labeled Seed + Unlabeled Pool', fontsize=12)
ax.legend(markerscale=2, fontsize=8, ncol=2)
ax.axis('off')
plt.tight_layout()
plt.savefig(f'{CFG["output_dir"]}/embeddings_{dim_method}.png', bbox_inches='tight', dpi=130)
plt.show()
print(f'Embedding visualization saved.')

## Section 13 â€” Uncertainty & Acquisition Scoring

In [ ]:
# â”€â”€ Compute uncertainty signals for every unlabeled image â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

@torch.no_grad()
def compute_uncertainty(model, dataset, batch_size=64, n_mc_passes=5):
    """
    Returns per-sample:
        probs      : (N, C)  mean softmax probabilities
        entropy    : (N,)    predictive entropy
        confidence : (N,)    max-probability (1 - uncertainty proxy)
        pred_class : (N,)    argmax of mean probs

    Uses MC-Dropout (n_mc_passes > 1) when available; falls back to
    deterministic single-pass if model has no dropout.
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=CFG['num_workers'],
                        pin_memory=torch.cuda.is_available())

    # Enable dropout for MC-Dropout style uncertainty (if any dropout exists)
    # ResNet-18 as defined has no dropout â€” we do n_mc_passes=1 but keep
    # the interface for potential future augmentation TTA.

    all_probs = []
    for _ in range(n_mc_passes):
        model.eval()   # BN in eval mode; no dropout in this arch
        pass_probs = []
        for batch in loader:
            if isinstance(batch, (list, tuple)):
                imgs = batch[0]
            else:
                imgs = batch
            imgs = imgs.to(DEVICE, non_blocking=True)
            with autocast(enabled=USE_AMP):
                logits = model(imgs)
            probs = F.softmax(logits, dim=1)
            pass_probs.append(probs.cpu().float().numpy())
        all_probs.append(np.concatenate(pass_probs, axis=0))

    mean_probs = np.mean(all_probs, axis=0)   # (N, C)
    # Predictive entropy: H = -sum(p * log(p))
    entropy     = -np.sum(mean_probs * np.log(mean_probs + 1e-12), axis=1)
    confidence  = mean_probs.max(axis=1)
    pred_class  = mean_probs.argmax(axis=1)
    return mean_probs, entropy, confidence, pred_class


print('Computing uncertainty scores on unlabeled pool...')
undef_probs, undef_entropy, undef_confidence, undef_pred_cls = compute_uncertainty(
    model_600, undef_ds_embed, batch_size=64
)
print(f'  Entropy  â€” mean={undef_entropy.mean():.3f}  max={undef_entropy.max():.3f}')
print(f'  Confid.  â€” mean={undef_confidence.mean():.3f}  min={undef_confidence.min():.3f}')
print(f'  Pred cls distribution: {Counter(undef_pred_cls.tolist())}')

## Section 14 â€” Active Learning / Data Selection

In [ ]:
# â”€â”€ Embedding-diversity score via k-nearest-neighbour distance â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.neighbors import NearestNeighbors

def compute_diversity_score(candidate_embs: np.ndarray,
                             selected_embs: np.ndarray,
                             k: int = 5) -> np.ndarray:
    """
    Diversity: for each candidate, compute mean distance to its k
    nearest neighbours in the ALREADY SELECTED set.
    High score â†’ far from already-selected samples â†’ more diverse.
    """
    if len(selected_embs) == 0:
        return np.ones(len(candidate_embs))
    k_actual = min(k, len(selected_embs))
    nbrs = NearestNeighbors(n_neighbors=k_actual, metric='cosine', algorithm='brute')
    nbrs.fit(selected_embs)
    dists, _ = nbrs.kneighbors(candidate_embs)
    return dists.mean(axis=1)   # (N,)


def compute_cluster_coverage_score(candidate_embs: np.ndarray,
                                   n_clusters: int = 30) -> np.ndarray:
    """
    K-means cluster the candidate pool; score each sample by how
    under-represented its cluster is.
    Samples in sparse clusters get higher scores.
    """
    from sklearn.cluster import MiniBatchKMeans
    km = MiniBatchKMeans(n_clusters=n_clusters, random_state=CFG['seed'],
                         batch_size=512, n_init=3)
    cluster_ids = km.fit_predict(candidate_embs)
    cluster_counts = np.bincount(cluster_ids, minlength=n_clusters)
    # Inverse frequency: rarer cluster â†’ higher score
    inv_freq = 1.0 / (cluster_counts[cluster_ids] + 1e-6)
    # Normalise to [0, 1]
    inv_freq = (inv_freq - inv_freq.min()) / (inv_freq.max() - inv_freq.min() + 1e-12)
    return inv_freq, cluster_ids


def compute_class_balance_score(pred_classes: np.ndarray,
                                 current_class_counts: dict,
                                 n_classes: int = 6) -> np.ndarray:
    """
    Samples predicted to belong to under-represented classes
    receive a higher score.
    """
    counts = np.array([current_class_counts.get(c, 0) for c in range(n_classes)], dtype=float)
    # Inverse count normalised
    inv_counts = 1.0 / (counts + 1.0)
    inv_counts /= inv_counts.sum()
    return inv_counts[pred_classes]   # (N,)


print('Acquisition utilities defined.')

In [ ]:
# â”€â”€ Master acquisition function â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def acquire_samples(
    model,
    pool_paths: list,
    pool_embs: np.ndarray,
    pool_probs: np.ndarray,
    pool_entropy: np.ndarray,
    pool_confidence: np.ndarray,
    pool_pred_cls: np.ndarray,
    already_selected_embs: np.ndarray,
    current_class_counts: dict,
    n_to_select: int,
    already_selected_indices: set,
    w_uncertainty: float = 0.35,
    w_diversity  : float = 0.35,
    w_cluster    : float = 0.15,
    w_balance    : float = 0.15,
):
    """
    Selects n_to_select indices from the pool using a composite
    acquisition score.

    Score = w_uncertainty * entropy_norm
          + w_diversity  * diversity_norm
          + w_cluster    * cluster_score
          + w_balance    * class_balance_score

    Returns:
        selected_indices  : list of int (indices into pool_paths)
        selected_paths    : list of str
        selected_pred_cls : list of int  (pseudo-labels)
    """
    # Mask out already-selected entries
    mask = np.ones(len(pool_paths), dtype=bool)
    for idx in already_selected_indices:
        mask[idx] = False
    candidate_idx = np.where(mask)[0]

    c_embs      = pool_embs[candidate_idx]
    c_entropy   = pool_entropy[candidate_idx]
    c_conf      = pool_confidence[candidate_idx]
    c_pred_cls  = pool_pred_cls[candidate_idx]

    # --- Uncertainty score (entropy, normalised) ---
    max_entropy = np.log(CFG['num_classes'])   # theoretical max
    unc_score   = c_entropy / (max_entropy + 1e-12)

    # --- Diversity score ---
    div_score = compute_diversity_score(c_embs, already_selected_embs)
    div_score = (div_score - div_score.min()) / (div_score.max() - div_score.min() + 1e-12)

    # --- Cluster coverage ---
    n_clust = min(30, len(candidate_idx) // 10 + 1)
    cluster_score, _ = compute_cluster_coverage_score(c_embs, n_clusters=n_clust)

    # --- Class balance ---
    bal_score = compute_class_balance_score(c_pred_cls, current_class_counts)
    bal_score = (bal_score - bal_score.min()) / (bal_score.max() - bal_score.min() + 1e-12)

    # --- Combined score ---
    combined = (w_uncertainty * unc_score
                + w_diversity   * div_score
                + w_cluster     * cluster_score
                + w_balance     * bal_score)

    # Select top-k
    top_k_local = np.argsort(-combined)[:n_to_select]
    selected_global = candidate_idx[top_k_local]

    selected_paths    = [pool_paths[i] for i in selected_global]
    selected_pred_cls = [int(pool_pred_cls[i]) for i in selected_global]

    return selected_global.tolist(), selected_paths, selected_pred_cls


print('Acquisition function defined.')

In [ ]:
# â”€â”€ Build the labeled sample registry â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# seed_ds_eval has transform=val_transform (clean, no augmentation)

# Initial seed samples list: (path, true_label)
seed_sample_list = list(seed_ds_eval.samples)   # [(path, label), ...]
print(f'Seed samples: {len(seed_sample_list)}')

# Current class counts from seed
current_class_counts_seed = Counter([lbl for _, lbl in seed_sample_list])
print(f'Seed class distribution: {dict(current_class_counts_seed)}')

# Embeddings of currently labeled set (seed)
selected_embs_so_far = seed_embs.copy()

# Track which pool indices have been selected
selected_pool_indices: set = set()

# Registry of all stages
AL_REGISTRY = {
    'stage_0': {
        'n_samples'  : len(seed_sample_list),
        'added_paths': [],
        'added_labels': [],
    }
}

print('AL registry initialised.')

## Section 15 â€” Stage 1 (600 â†’ ~1400 samples)

In [ ]:
# â”€â”€ Stage 1 acquisition: select ~800 more from pool â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
N_STAGE1 = 1400
n_to_add_stage1 = N_STAGE1 - len(seed_sample_list)   # ~800
print(f'Stage 1: acquiring {n_to_add_stage1} new samples from pool...')

s1_idx, s1_paths, s1_pred_labels = acquire_samples(
    model=model_600,
    pool_paths=undef_paths,
    pool_embs=undef_embs,
    pool_probs=undef_probs,
    pool_entropy=undef_entropy,
    pool_confidence=undef_confidence,
    pool_pred_cls=undef_pred_cls,
    already_selected_embs=selected_embs_so_far,
    current_class_counts=dict(current_class_counts_seed),
    n_to_select=n_to_add_stage1,
    already_selected_indices=selected_pool_indices,
)

selected_pool_indices.update(s1_idx)
selected_embs_so_far = np.concatenate([selected_embs_so_far, undef_embs[s1_idx]], axis=0)

# Build combined training set (seed + stage-1)
stage1_new_samples  = list(zip(s1_paths, s1_pred_labels))
combined_samples_1400 = seed_sample_list + stage1_new_samples
print(f'Stage 1 total training set: {len(combined_samples_1400)}')

# Class distribution of newly acquired samples
new_cls_dist = Counter(s1_pred_labels)
print(f'  New samples by predicted class: {dict(new_cls_dist)}')

# Update class counts
current_class_counts_1400 = Counter([lbl for _, lbl in combined_samples_1400])

AL_REGISTRY['stage_1'] = {
    'n_samples'   : len(combined_samples_1400),
    'added_paths' : s1_paths,
    'added_labels': s1_pred_labels,
}

# Train
set_seed(CFG['seed'])
train_ds_1400 = CombinedLabeledDataset(combined_samples_1400, transform=train_transform)
train_loader_1400 = DataLoader(train_ds_1400, batch_size=CFG['batch_size'],
                                shuffle=True, **LOADER_KW)

model_1400 = make_model()
# Warm-start from stage-0 best weights
model_1400.load_state_dict(torch.load(save_600, map_location=DEVICE))

save_1400 = f'{CFG["output_dir"]}/model_1400.pth'
best_acc_1400, best_f1_1400, history_1400 = full_train(
    model_1400, train_loader_1400, val_loader,
    epochs=CFG['epochs_finetune'],
    lr=CFG['lr'] * 0.5,
    weight_decay=CFG['weight_decay'],
    save_path=save_1400,
    patience=CFG['patience'],
    stage_label='Stage-1 | 1400 samples',
)
plot_history(history_1400, title='Stage 1 (1400)', save_path=f'{CFG["output_dir"]}/history_1400.png')

## Section 16 â€” Stage 2 (~1400 â†’ ~2200 samples)

In [ ]:
# â”€â”€ Stage 2: re-score pool with the improved stage-1 model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('Re-extracting embeddings & uncertainty with Stage-1 model...')
model_1400.load_state_dict(torch.load(save_1400, map_location=DEVICE))

undef_embs_2, _, _ = extract_embeddings(model_1400, undef_ds_embed, return_paths=True)
undef_probs_2, undef_entropy_2, undef_confidence_2, undef_pred_cls_2 = compute_uncertainty(
    model_1400, undef_ds_embed, batch_size=64
)

N_STAGE2    = 2200
n_to_add_s2 = N_STAGE2 - len(combined_samples_1400)
print(f'Stage 2: acquiring {n_to_add_s2} more samples...')

s2_idx, s2_paths, s2_pred_labels = acquire_samples(
    model=model_1400,
    pool_paths=undef_paths,
    pool_embs=undef_embs_2,
    pool_probs=undef_probs_2,
    pool_entropy=undef_entropy_2,
    pool_confidence=undef_confidence_2,
    pool_pred_cls=undef_pred_cls_2,
    already_selected_embs=selected_embs_so_far,
    current_class_counts=dict(current_class_counts_1400),
    n_to_select=n_to_add_s2,
    already_selected_indices=selected_pool_indices,
)

selected_pool_indices.update(s2_idx)
selected_embs_so_far = np.concatenate([selected_embs_so_far, undef_embs_2[s2_idx]], axis=0)

stage2_new_samples    = list(zip(s2_paths, s2_pred_labels))
combined_samples_2200 = combined_samples_1400 + stage2_new_samples
print(f'Stage 2 total training set: {len(combined_samples_2200)}')

current_class_counts_2200 = Counter([lbl for _, lbl in combined_samples_2200])

AL_REGISTRY['stage_2'] = {
    'n_samples'   : len(combined_samples_2200),
    'added_paths' : s2_paths,
    'added_labels': s2_pred_labels,
}

set_seed(CFG['seed'])
train_ds_2200 = CombinedLabeledDataset(combined_samples_2200, transform=train_transform)
train_loader_2200 = DataLoader(train_ds_2200, batch_size=CFG['batch_size'],
                                shuffle=True, **LOADER_KW)

model_2200 = make_model()
model_2200.load_state_dict(torch.load(save_1400, map_location=DEVICE))

save_2200 = f'{CFG["output_dir"]}/model_2200.pth'
best_acc_2200, best_f1_2200, history_2200 = full_train(
    model_2200, train_loader_2200, val_loader,
    epochs=CFG['epochs_finetune'],
    lr=CFG['lr'] * 0.3,
    weight_decay=CFG['weight_decay'],
    save_path=save_2200,
    patience=CFG['patience'],
    stage_label='Stage-2 | 2200 samples',
)
plot_history(history_2200, title='Stage 2 (2200)', save_path=f'{CFG["output_dir"]}/history_2200.png')

## Section 17 â€” Stage 3 (~2200 â†’ 3000 samples)

In [ ]:
# â”€â”€ Stage 3: final acquisition up to exactly 3000 samples â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('Re-extracting embeddings & uncertainty with Stage-2 model...')
model_2200.load_state_dict(torch.load(save_2200, map_location=DEVICE))

undef_embs_3, _, _ = extract_embeddings(model_2200, undef_ds_embed, return_paths=True)
undef_probs_3, undef_entropy_3, undef_confidence_3, undef_pred_cls_3 = compute_uncertainty(
    model_2200, undef_ds_embed, batch_size=64
)

N_STAGE3    = CFG['max_samples']   # exactly 3000
n_to_add_s3 = N_STAGE3 - len(combined_samples_2200)
print(f'Stage 3: acquiring final {n_to_add_s3} samples (total will be {N_STAGE3})...')

s3_idx, s3_paths, s3_pred_labels = acquire_samples(
    model=model_2200,
    pool_paths=undef_paths,
    pool_embs=undef_embs_3,
    pool_probs=undef_probs_3,
    pool_entropy=undef_entropy_3,
    pool_confidence=undef_confidence_3,
    pool_pred_cls=undef_pred_cls_3,
    already_selected_embs=selected_embs_so_far,
    current_class_counts=dict(current_class_counts_2200),
    n_to_select=n_to_add_s3,
    already_selected_indices=selected_pool_indices,
)

selected_pool_indices.update(s3_idx)
selected_embs_so_far = np.concatenate([selected_embs_so_far, undef_embs_3[s3_idx]], axis=0)

stage3_new_samples    = list(zip(s3_paths, s3_pred_labels))
combined_samples_3000 = combined_samples_2200 + stage3_new_samples
assert len(combined_samples_3000) <= CFG['max_samples'], \
    f'Constraint violated: {len(combined_samples_3000)} > {CFG["max_samples"]}'
print(f'Stage 3 total training set: {len(combined_samples_3000)}  (cap={CFG["max_samples"]})')

current_class_counts_3000 = Counter([lbl for _, lbl in combined_samples_3000])
print(f'Final class distribution: {dict(current_class_counts_3000)}')

AL_REGISTRY['stage_3'] = {
    'n_samples'   : len(combined_samples_3000),
    'added_paths' : s3_paths,
    'added_labels': s3_pred_labels,
}

set_seed(CFG['seed'])
train_ds_3000 = CombinedLabeledDataset(combined_samples_3000, transform=train_transform)
train_loader_3000 = DataLoader(train_ds_3000, batch_size=CFG['batch_size'],
                                shuffle=True, **LOADER_KW)

model_3000 = make_model()
model_3000.load_state_dict(torch.load(save_2200, map_location=DEVICE))

save_3000 = f'{CFG["output_dir"]}/model_3000.pth'
best_acc_3000, best_f1_3000, history_3000 = full_train(
    model_3000, train_loader_3000, val_loader,
    epochs=CFG['epochs_finetune'],
    lr=CFG['lr'] * 0.2,
    weight_decay=CFG['weight_decay'],
    save_path=save_3000,
    patience=CFG['patience'],
    stage_label='Stage-3 | 3000 samples',
)
plot_history(history_3000, title='Stage 3 (3000)', save_path=f'{CFG["output_dir"]}/history_3000.png')

## Section 18 â€” Model Comparison

In [ ]:
# â”€â”€ Compare all stages â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
stage_results = [
    {'Dataset Size': 600,  'Val Accuracy': best_acc_600,  'Macro F1': best_f1_600,  'Model': save_600},
    {'Dataset Size': 1400, 'Val Accuracy': best_acc_1400, 'Macro F1': best_f1_1400, 'Model': save_1400},
    {'Dataset Size': 2200, 'Val Accuracy': best_acc_2200, 'Macro F1': best_f1_2200, 'Model': save_2200},
    {'Dataset Size': 3000, 'Val Accuracy': best_acc_3000, 'Macro F1': best_f1_3000, 'Model': save_3000},
]

results_df = pd.DataFrame(stage_results)
results_df['Val Accuracy %'] = (results_df['Val Accuracy'] * 100).round(2)
results_df['Macro F1 %']     = (results_df['Macro F1']     * 100).round(2)

print('\n=== Active Learning Progress ===')
print(results_df[['Dataset Size', 'Val Accuracy %', 'Macro F1 %']].to_string(index=False))

# Plot accuracy vs data size
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results_df['Dataset Size'], results_df['Val Accuracy'], marker='o', linewidth=2, label='Val Accuracy')
ax.plot(results_df['Dataset Size'], results_df['Macro F1'],     marker='s', linewidth=2, label='Macro F1')
for _, row in results_df.iterrows():
    ax.annotate(f"{row['Val Accuracy %']:.1f}%",
                (row['Dataset Size'], row['Val Accuracy']),
                textcoords='offset points', xytext=(5, 5), fontsize=9)
ax.set_xlabel('Training Dataset Size')
ax.set_ylabel('Score')
ax.set_title('Active Learning: Accuracy vs Data Size')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CFG["output_dir"]}/al_progress.png', bbox_inches='tight', dpi=120)
plt.show()

# Select best model based ONLY on validation accuracy
best_row   = results_df.loc[results_df['Val Accuracy'].idxmax()]
best_model_path = best_row['Model']
print(f'\nBest model: {best_model_path}')
print(f'Best val accuracy: {best_row["Val Accuracy %"]}%')

## Section 19 â€” 3LC Integration

In [ ]:
# â”€â”€ 3LC Integration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 3LC is the core tool for this competition (Train â†’ Analyze â†’ Fix â†’ Retrain).
# If 3LC credentials are not configured in this Kaggle environment, the block
# below runs in OFFLINE mode and documents where integration would hook in.
#
# To enable full 3LC:
#  1. Create an account at https://account.3lc.ai
#  2. Get your API key from https://account.3lc.ai/api-key
#  3. In a local or authenticated Kaggle environment run:
#       3lc login <your_api_key>
#       3lc service
#  4. Then re-run this cell â€” the TLC_AVAILABLE flag will flip to True.
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

TLC_AVAILABLE = False

try:
    import tlc
    # Attempt a lightweight call to confirm service connectivity
    _url = tlc.Url('tlc://test')
    TLC_AVAILABLE = True
    print(f'3LC version : {tlc.__version__}')
    print('3LC service : connected')
except Exception as e:
    print(f'3LC not connected ({e}). Running in local-fallback mode.')
    print('All data-centric analysis is still performed locally.')


if TLC_AVAILABLE:
    # â”€â”€ Register dataset tables â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # This mirrors register_tables.py from the starter kit.

    PROJECT_NAME = 'Intel-Scene-HackBlox'

    def register_image_folder_table(root: Path, table_name: str, split: str):
        """Register an ImageFolder-style split with 3LC."""
        samples = []
        for cls_idx, cls in enumerate(CLASS_NAMES):
            cls_dir = root / cls
            if not cls_dir.is_dir():
                continue
            for img in sorted(cls_dir.glob('*.jpg')):
                samples.append({'image': str(img), 'label': cls_idx, 'weight': 1})

        schema = tlc.Schema({
            'image': tlc.ImageUrlStringValue(),
            'label': tlc.Int32Value(
                value_min=0, value_max=5,
                value_map={str(i): c for i, c in enumerate(CLASS_NAMES)}
            ),
            'weight': tlc.Float32Value(),
        })

        table = tlc.Table.from_list(
            samples,
            schema=schema,
            table_name=table_name,
            project_name=PROJECT_NAME,
        )
        print(f'Registered table: {table_name} ({len(samples)} rows) â€” {table.url}')
        return table

    train_table = register_image_folder_table(TRAIN_DIR, 'train', 'train')
    val_table   = register_image_folder_table(VAL_DIR,   'val',   'val')

    # â”€â”€ Log a training run with embeddings â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Load the best final model
    final_model_for_tlc = make_model()
    final_model_for_tlc.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    final_model_for_tlc.eval()

    run = tlc.Run.create(
        project_name=PROJECT_NAME,
        run_name='hackblox-active-learning-final',
        description='ResNet-18 from scratch, 3-stage active learning, 3000 samples',
    )

    # Log val embeddings + metrics per sample
    val_embs_final, val_labels_final = extract_embeddings(final_model_for_tlc, val_ds_embed)
    criterion_eval2 = nn.CrossEntropyLoss()
    _, _, _, val_preds_final, _ = evaluate(final_model_for_tlc, val_loader, criterion_eval2)

    for i, (emb, label, pred) in enumerate(zip(val_embs_final, val_labels_final, val_preds_final)):
        run.log_metrics({
            'val_pred'   : int(pred),
            'val_true'   : int(label),
            'is_correct' : int(pred == label),
            'embedding'  : emb.tolist(),
        }, sample_index=i)

    run.set_status_completed()
    print(f'3LC run logged: {run.url}')
    print('View in 3LC Dashboard â†’ see embeddings, per-sample metrics, data quality.')

else:
    # â”€â”€ Local fallback: same analysis without 3LC service â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print('\n[LOCAL FALLBACK] â€” 3LC methodology implemented locally:')
    print('  1. TRAIN:    Baseline on 600 labeled images.')
    print('  2. ANALYSE:  Extract embeddings, compute uncertainty/entropy/confidence.')
    print('  3. FIX DATA: Acquisition function selects informative samples from pool.')
    print('  4. RETRAIN:  3 progressive stages (600â†’1400â†’2200â†’3000).')
    print('  5. SUBMIT:   Best model by val accuracy â†’ submission.csv.')
    print()
    print('  To enable 3LC Dashboard visualisation:')
    print('    pip install "3lc==2.22.3"')
    print('    3lc login <api_key>')
    print('    3lc service')
    print('  Then re-run this cell.')

## Section 20 â€” Embedding Visualization (Post Active-Learning)

In [ ]:
# â”€â”€ Visualise: selected vs non-selected pool samples â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Re-use the stage-1 undef embeddings for a representative plot
# (using a fixed subsample for speed)

N_POOL_VIZ = min(1500, len(undef_embs))
viz_idx = rng.choice(len(undef_embs), N_POOL_VIZ, replace=False)

viz_embs   = np.concatenate([seed_embs, undef_embs[viz_idx]], axis=0)
viz_labels = np.concatenate([seed_labels, np.full(N_POOL_VIZ, -1)])
viz_selected = np.array([i in selected_pool_indices for i in viz_idx], dtype=bool)

viz_reduced, viz_method = reduce_embeddings(viz_embs, method='auto')

fig, ax = plt.subplots(figsize=(11, 8))
n_seed_v = len(seed_embs)

# Unselected pool
not_sel_mask = ~viz_selected
ax.scatter(viz_reduced[n_seed_v:][not_sel_mask, 0],
           viz_reduced[n_seed_v:][not_sel_mask, 1],
           c='lightgray', s=6, alpha=0.3, label='Pool (not selected)')

# Selected pool
ax.scatter(viz_reduced[n_seed_v:][viz_selected, 0],
           viz_reduced[n_seed_v:][viz_selected, 1],
           c='orange', s=12, alpha=0.6, label='Pool (selected by AL)', marker='^')

# Labeled seed by class
for i, cls in enumerate(CLASS_NAMES):
    mask = seed_labels == i
    ax.scatter(viz_reduced[:n_seed_v][mask, 0], viz_reduced[:n_seed_v][mask, 1],
               c=[COLORS[i]], s=30, alpha=0.9, label=cls, zorder=3)

ax.set_title(f'{viz_method.upper()} â€” Active Learning Selection Overlay', fontsize=12)
ax.legend(markerscale=2, fontsize=7, ncol=3, loc='upper right')
ax.axis('off')
plt.tight_layout()
plt.savefig(f'{CFG["output_dir"]}/al_selection_{viz_method}.png', bbox_inches='tight', dpi=130)
plt.show()

# â”€â”€ Uncertainty distribution plot â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(undef_entropy, bins=60, color='steelblue', alpha=0.7, label='All pool')
sel_entropy = undef_entropy[list(selected_pool_indices)]
ax.hist(sel_entropy, bins=60, color='orange', alpha=0.7, label='Selected')
ax.set_xlabel('Entropy')
ax.set_ylabel('Count')
ax.set_title('Uncertainty Distribution â€” Pool vs Selected Samples')
ax.legend()
plt.tight_layout()
plt.savefig(f'{CFG["output_dir"]}/uncertainty_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

## Section 21 â€” Final Error Analysis

In [ ]:
# â”€â”€ Load the best model for analysis â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
final_model = make_model()
final_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
final_model.eval()
print(f'Final model loaded from: {best_model_path}')

# Full validation-set predictions with paths + confidence
@torch.no_grad()
def predict_with_metadata(model, dataset):
    loader = DataLoader(dataset, batch_size=64, shuffle=False, **LOADER_KW)
    all_paths, all_true, all_pred, all_conf, all_ent = [], [], [], [], []
    model.eval()
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
        probs = F.softmax(logits, dim=1).cpu().float().numpy()
        preds = probs.argmax(axis=1)
        confs = probs.max(axis=1)
        ents  = -np.sum(probs * np.log(probs + 1e-12), axis=1)
        all_pred.extend(preds.tolist())
        all_conf.extend(confs.tolist())
        all_ent.extend(ents.tolist())
        all_true.extend(labels.tolist())
    return all_true, all_pred, all_conf, all_ent


val_true, val_pred, val_conf, val_ent = predict_with_metadata(final_model, val_ds)

error_df = pd.DataFrame({
    'image_path'     : [s[0] for s in val_ds.samples],
    'true_label'     : val_true,
    'predicted_label': val_pred,
    'true_name'      : [CLASS_NAMES[t] for t in val_true],
    'pred_name'      : [CLASS_NAMES[p] for p in val_pred],
    'confidence'     : val_conf,
    'entropy'        : val_ent,
    'correct'        : [t == p for t, p in zip(val_true, val_pred)],
})

errors = error_df[~error_df['correct']].sort_values('confidence', ascending=False)
print(f'Total val errors: {len(errors)} / {len(error_df)}  '
      f'({100*len(errors)/len(error_df):.1f}%)')

# â”€â”€ Top high-confidence errors â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\n--- Top 10 High-Confidence Wrong Predictions ---')
print(errors[['true_name', 'pred_name', 'confidence', 'entropy']].head(10).to_string(index=False))

# â”€â”€ Low-confidence predictions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
low_conf = error_df.sort_values('confidence').head(10)
print('\n--- Top 10 Lowest-Confidence Predictions ---')
print(low_conf[['true_name', 'pred_name', 'confidence', 'entropy', 'correct']].to_string(index=False))

In [ ]:
# â”€â”€ Final confusion matrix + class analysis â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
cm_final = plot_confusion_matrix(
    val_true, val_pred,
    title='Final Model â€” Confusion Matrix (Validation)',
    save_path=f'{CFG["output_dir"]}/cm_final.png',
)
print_classification_report(val_true, val_pred)

# Per-class accuracy
print('\nPer-class accuracy:')
for i, cls in enumerate(CLASS_NAMES):
    cls_mask = np.array(val_true) == i
    cls_acc  = np.mean(np.array(val_pred)[cls_mask] == i)
    print(f'  {cls:12s}: {cls_acc:.4f}')

# â”€â”€ Identify most confused class pairs â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\nMost confused class pairs:')
off_diag = []
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        if i != j and cm_final[i, j] > 0:
            off_diag.append((CLASS_NAMES[i], CLASS_NAMES[j], int(cm_final[i, j])))
off_diag.sort(key=lambda x: -x[2])
for true_cls, pred_cls, count in off_diag[:8]:
    print(f'  True={true_cls:12s} â†’ Predicted={pred_cls:12s} : {count}')

print('''
Analysis:
  â€¢ glacier â†” mountain: both have cold-blue palettes and snow/rock textures,
    making them the hardest pair to separate with limited training data.
  â€¢ mountain â†” forest: low-altitude mountain shots often include dense tree cover.
  â€¢ sea â†” glacier: both have large homogeneous blue/white regions.
  â€¢ buildings â†” street: urban images frequently contain both elements.
''')

In [ ]:
# â”€â”€ Visualise most-confused samples â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
if len(errors) >= 6:
    top_errors = errors.head(6)
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, (_, row) in zip(axes.flat, top_errors.iterrows()):
        try:
            img = Image.open(row['image_path'])
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(
            f'True: {row["true_name"]}\nPred: {row["pred_name"]}\nConf: {row["confidence"]:.2f}',
            fontsize=8
        )
        ax.axis('off')
    plt.suptitle('High-Confidence Errors â€” Final Model', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{CFG["output_dir"]}/error_analysis.png', bbox_inches='tight', dpi=120)
    plt.show()

## Section 22 â€” Final Test Prediction

In [ ]:
# â”€â”€ Generate predictions on the 1800 hidden test images â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# IMPORTANT: test images are NEVER used for training, validation, or model selection.

@torch.no_grad()
def predict_test(model, test_dataset):
    loader = DataLoader(test_dataset, batch_size=64, shuffle=False, **LOADER_KW)
    model.eval()
    image_ids, predictions, confidences = [], [], []
    for imgs, img_ids in tqdm(loader, desc='Test inference'):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
        probs = F.softmax(logits, dim=1).cpu().float().numpy()
        preds = probs.argmax(axis=1)
        confs = probs.max(axis=1)
        image_ids.extend(list(img_ids))
        predictions.extend(preds.tolist())
        confidences.extend(confs.tolist())
    return image_ids, predictions, confidences


test_ids, test_preds, test_confs = predict_test(final_model, test_ds)

print(f'Test predictions generated: {len(test_ids)}')
print(f'Prediction class distribution: {Counter(test_preds)}')

## Section 23 â€” Submission Validation & Save

In [ ]:
# â”€â”€ Build submission dataframe â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
submission = pd.DataFrame({
    'image_id'  : test_ids,
    'prediction': [int(p) for p in test_preds],
    'confidence': [float(round(c, 6)) for c in test_confs],
})

# â”€â”€ Align with sample_submission.csv if available â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
sample_sub_paths = list(DATASET_ROOT.glob('sample_submission.csv'))
if not sample_sub_paths:
    sample_sub_paths = list(Path('/kaggle/input').rglob('sample_submission.csv'))

if sample_sub_paths:
    sample_sub = pd.read_csv(sample_sub_paths[0])
    # Merge to preserve reference order and catch any missing IDs
    submission = sample_sub[['image_id']].merge(
        submission, on='image_id', how='left'
    )
    # Fill any gaps (shouldn't happen) with class 0 and confidence 0.5
    submission['prediction'] = submission['prediction'].fillna(0).astype(int)
    submission['confidence'] = submission['confidence'].fillna(0.5)
    print(f'Aligned to sample_submission.csv ({len(submission)} rows).')
else:
    print('sample_submission.csv not found â€” using predicted order.')

print(submission.head(10))

In [ ]:
# â”€â”€ Validation checks â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
EXPECTED_ROWS = 1800

checks = {}

# 1. Row count
checks['row_count_ok']   = len(submission) == EXPECTED_ROWS
# 2. No duplicate image IDs
checks['no_duplicates']  = submission['image_id'].nunique() == len(submission)
# 3. Predictions are integers 0â€“5
checks['valid_preds']    = submission['prediction'].between(0, 5).all()
checks['int_preds']      = submission['prediction'].dtype in [int, 'int64', 'int32']
# 4. Confidence in [0, 1]
checks['valid_conf']     = submission['confidence'].between(0.0, 1.0).all()
# 5. No NaN
checks['no_nan']         = not submission.isnull().any().any()
# 6. Column names
checks['columns_ok']     = list(submission.columns) == ['image_id', 'prediction', 'confidence']
# 7. image_ids match test folder
test_stems = {Path(p).stem for p in test_ds.paths}
sub_ids    = set(submission['image_id'].tolist())
checks['ids_match_test'] = sub_ids == test_stems or sub_ids.issubset(test_stems)

print('\n=== Submission Validation ===')
all_passed = True
for check, result in checks.items():
    status = 'âœ“' if result else 'âœ—'
    if not result:
        all_passed = False
    print(f'  [{status}] {check}')

print(f'\nRows     : {len(submission)}')
print(f'Unique ID: {submission["image_id"].nunique()}')
print(f'Pred dist: {dict(Counter(submission["prediction"].tolist()))}')
print(f'Conf range: [{submission["confidence"].min():.4f}, {submission["confidence"].max():.4f}]')

# â”€â”€ Save submission â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
sub_path        = f'{CFG["output_dir"]}/submission.csv'
sub_backup_path = f'{CFG["output_dir"]}/submission_final.csv'

submission.to_csv(sub_path,        index=False)
submission.to_csv(sub_backup_path, index=False)
print(f'\nSaved: {sub_path}')
print(f'Saved: {sub_backup_path}')

print('\nFirst 10 rows:')
print(submission.head(10).to_string(index=False))

if all_passed:
    print('\n' + '='*50)
    print('          SUBMISSION READY')
    print('='*50)
else:
    print('\n[WARNING] Some validation checks failed. Review above.')

## Section 24 â€” Save Artifacts & Summary

In [ ]:
# â”€â”€ Save AL registry â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
al_summary = {}
for stage, info in AL_REGISTRY.items():
    al_summary[stage] = {
        'n_samples'   : info['n_samples'],
        'n_added'     : len(info['added_paths']),
        'class_dist'  : dict(Counter(info['added_labels'])) if info['added_labels'] else {},
    }

with open(f'{CFG["output_dir"]}/al_registry.json', 'w') as f:
    json.dump(al_summary, f, indent=2)

# â”€â”€ Save config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
with open(f'{CFG["output_dir"]}/run_config.json', 'w') as f:
    json.dump(CFG, f, indent=2)

# â”€â”€ Final summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('='*60)
print(' HackBlox 2026 â€” Final Run Summary')
print('='*60)
print(f' Model        : ResNet-18 from scratch (no pretrained weights)')
print(f' Best model   : {best_model_path}')
print(f' Val accuracy : {best_row["Val Accuracy %"]}%')
print(f' Macro F1     : {best_row["Macro F1 %"]}%')
print(f' Training cap : {len(combined_samples_3000)}/{CFG["max_samples"]} samples')
print(f' 3LC active   : {TLC_AVAILABLE}')
print()
print(' Active Learning stages:')
for stage, info in al_summary.items():
    print(f'   {stage}: {info["n_samples"]} total, +{info["n_added"]} added')
print()
print(' Output files:')
for fname in sorted(Path(CFG['output_dir']).glob('*')):
    print(f'   {fname.name}')
print('='*60)